In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/processed_logs.csv')


In [2]:
drop_cols = [
    'entity_id', 'name', 'email', 'source_ip', 'mac_address',
    'device_fingerprint', 'command_sequence', 'timestamp', 'logout_time',
    'attack_type',
    'device_id_x', 'device_id_y',   # <-- add these (duplicated from a merge)
    'normal_login_end', 'normal_session_duration',
    'normal_entity_type', 'normal_department', 'normal_geo_location',
    'normal_device_id', 'normal_device_fingerprint', 'normal_auth_method',
    'normal_resources',
]

df_model = df.drop(columns=[c for c in drop_cols if c in df.columns])

In [3]:
bool_cols = ['location_changed', 'device_changed', 'auth_changed',
             'login_time_changed', 'long_session', 'high_failed_login',
             'resource_changed']

df_model[bool_cols] = df_model[bool_cols].astype(int)

In [4]:
df_model.head()

,entity_type,geo_location,resource_accessed,auth_method,session_duration,failed_login_attempts,label,department,office,device_type,...,normal_failed_login_attempts,login_hour,location_changed,device_changed,auth_changed,login_time_changed,long_session,high_failed_login,resource_changed,risk_score
0,user,Bangalore,Internal Server,Password,76,0,Normal,Marketing,Bangalore,Workstation,...,0,8,0,0,0,0,0,0,0,0
1,user,Pune,CRM,MFA,118,2,Normal,Support,Pune,Desktop,...,1,9,0,0,0,0,0,1,0,1
2,user,Bangalore,Confluence,Biometric,99,0,Normal,Marketing,Bangalore,Desktop,...,0,10,0,0,0,0,0,0,0,0
3,user,Bangalore,Jira,Password,64,0,Normal,Support,Bangalore,Workstation,...,0,11,0,0,0,0,0,0,0,0
4,user,Pune,Finance DB,MFA,143,0,Normal,Marketing,Pune,Desktop,...,0,10,0,0,0,0,0,0,0,0


In [5]:
df_model.columns

Index(['entity_type', 'geo_location', 'resource_accessed', 'auth_method',
       'session_duration', 'failed_login_attempts', 'label', 'department',
       'office', 'device_type', 'operating_system', 'browser',
       'normal_login_start', 'normal_failed_login_attempts', 'login_hour',
       'location_changed', 'device_changed', 'auth_changed',
       'login_time_changed', 'long_session', 'high_failed_login',
       'resource_changed', 'risk_score'],
      dtype='str')

In [6]:
from sklearn.preprocessing import LabelEncoder
import joblib

categorical_cols = ['geo_location', 'resource_accessed', 'auth_method',
                     'device_type', 'operating_system', 'browser',
                     'department', 'office', 'entity_type']

encoders = {}
for col in categorical_cols:
    if col in df_model.columns:
        le = LabelEncoder()
        df_model[col] = le.fit_transform(df_model[col].astype(str))
        encoders[col] = le

joblib.dump(encoders, '../trained_models/label_encoder.pkl')

['../trained_models/label_encoder.pkl']

In [7]:
from sklearn.preprocessing import StandardScaler

numeric_cols = ['session_duration', 'failed_login_attempts', 'login_hour',
                 'normal_login_start', 'normal_failed_login_attempts', 'risk_score']

scaler = StandardScaler()
df_model[numeric_cols] = scaler.fit_transform(df_model[numeric_cols])

joblib.dump(scaler, '../trained_models/scaler.pkl')

['../trained_models/scaler.pkl']

In [8]:
X = df_model.drop(columns=['label'])
y = df_model['label']  # Normal / Attack

joblib.dump(list(X.columns), '../trained_models/feature_columns.pkl')

# Save processed dataset
df_model.to_csv('../data/processed/processed_logs.csv', index=False)